# RodPrep — exploration

Notebook d'exploration de l'étape 1 : extraction du récap Excel ROD et construction de la table hôtel.

Objectif : visualiser les entrées, les étapes intermédiaires et remplir `../Output/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "RodPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

## 1. Entrée — récap Excel et registre identité

In [ ]:
from rod_ia.config.settings import get_settings
from rod_ia.domain.repositories.identity_registry import HotelIdentityRegistry
from rod_prep.prep import RodPrep

settings = get_settings(PROJECT)
prep = RodPrep(INPUT_DIR, OUTPUT_DIR, settings.identity_registry_path)

recap_path = prep.seed_input_from_sources()
print("Fichier récap :", recap_path)

registry = HotelIdentityRegistry(settings.identity_registry_path)
registry_df = pd.DataFrame([r.to_dict() for r in registry.all_records()])
print(f"Registre identité : {len(registry_df)} hôtels")
registry_df[["hotel_id", "name_ventes", "brand", "city", "nb_chambres"]].head(10)

## 2. Extraction longue — une ligne par variable × hôtel

In [ ]:
from rod_ia.domain.services.rod_recap_extractor import RodRecapExtractor

extractor = RodRecapExtractor(
    recap_path=recap_path,
    identity_registry=registry,
    output_path=OUTPUT_DIR / "rod_recap",
)

long_df = extractor.extract_long()
print(f"Format long : {long_df.shape[0]} lignes × {long_df.shape[1]} colonnes")
long_df.head(12)

## 3. Format wide — features `d_recap_*` par hôtel

In [ ]:
wide_df = extractor.extract_wide()
print(f"Format wide : {wide_df.shape[0]} hôtels × {wide_df.shape[1]} colonnes")
wide_df.head()

## 4. Table de liaison `hotel_lookup`

In [ ]:
hotel_lookup = prep.run()  # persiste aussi rod_features + hotel_lookup
print(f"hotel_lookup : {hotel_lookup.shape}")
hotel_lookup.head()

## 5. Aperçu colonnes récap retenues

In [ ]:
recap_cols = [c for c in hotel_lookup.columns if str(c).startswith("d_recap_")]
print(f"{len(recap_cols)} colonnes d_recap_")
if recap_cols:
    hotel_lookup[["hotel_code", "nom_hotel"] + recap_cols[:8]].head()

## 6. Entrée MeteoPrep / ProximityPrep

In [ ]:
meteo_input = prep.to_meteo_input()
meteo_input

## 7. Fichiers produits dans Output/

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)